In [2]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from xgboost import XGBClassifier
from sklearn.base import clone
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import roc_curve, auc, accuracy_score

## Time-specific classifiers:

### This section trains time-specific (3 possible timepoints, 3 tissues) classifiers for chromatin loop formation prediction

1. Trains cross-validated classifiers and plots their ROC curves (with mean and std)
2. For each time window: generates a combined plot with ROC curves of the best CV classifier for each tissue
3. For each configuration of (timepoint, tissue) it saves two models:
    - best cross-validation classifier
    - classifier fitted to the entire data

In [2]:
def time_specific_with_cv_and_roc_plots(classifier, windows=["06-08", "10-12", "14-16"], tissues=["Neuroblasts", "Neurons", "Glia"], n_splits=10):

    def make_names_dict():
        # FIXME: move this to some kind of config
        # Dictionary for f-string construction in model paths and in text on ROC plots
        model_names = ['RandomForestClassifier', 'SVC', 'LogisticRegression', 'XGBClassifier']
        model_names_dict = {name:{'full':'', 'short':''} for name in model_names}

        model_names_dict['RandomForestClassifier']['full'] = 'Random Forest'
        model_names_dict['RandomForestClassifier']['short'] = 'RF'

        model_names_dict['SVC']['full'] = 'Support Vector Machine'
        model_names_dict['SVC']['short'] = 'SVM'

        model_names_dict['LogisticRegression']['full'] = 'Logistic Regression'
        model_names_dict['LogisticRegression']['short'] = 'LR'

        model_names_dict['XGBClassifier']['full'] = 'XGBoost'
        model_names_dict['XGBClassifier']['short'] = 'XGB'

        # type(model).__name__
        return model_names_dict

    model_names_dict = make_names_dict()
    model_name = model_names_dict[type(classifier).__name__]['full']
    short = model_names_dict[type(classifier).__name__]['short']

    for w in windows:
        
        X = pd.read_csv(f"data/training/hrs{w}/data_diff_hrs{w}.csv", index_col=0)
        
        # Prepare the per-window ROC figure
        #plt.figure(figsize=(8, 8))
        tissue_plotting_info = {t: {} for t in tissues}

        for t in tissues:
            y_path = f"data/training/hrs{w}/y_{t}.csv"
            if not os.path.exists(y_path):
                continue
            y = pd.read_csv(y_path, index_col=0).iloc[:, 0]

            # Align X and y just in case
            X = X.loc[y.index]

            # Cross-validation
            kf = KFold(n_splits=n_splits, shuffle=True, random_state=0)
            probs, trues, accs = [], [], []
            fold_index = {}

            for (i, (train_idx, test_idx)) in enumerate(kf.split(X)):
                X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
                y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
                fold_index[i] = (train_idx, test_idx)

                classifier = clone(classifier) # this creates an UNFITTED model with the same parameters as before
                classifier.fit(X_train, y_train)

                p = classifier.predict_proba(X_test)[:, 1]
                probs.append(p)
                trues.append(y_test.values)
                accs.append(accuracy_score(y_test, (p > 0.5).astype(int)))

            # Compute ROC curve and other metrics
            mean_acc = np.mean(accs)
            std_acc = np.std(accs)

            fprs, tprs, roc_aucs = [], [], []
            for (true, prob) in zip(trues, probs):
                fpr, tpr, _ = roc_curve(true, prob)
                fprs.append(fpr)
                tprs.append(tpr)
                roc_aucs.append(auc(fpr, tpr))

            # Use linear interpolation to plot mean +/- 1 std. bounds
            mean_fpr = np.linspace(0, 1, 100)
            interp_tprs = []

            _, ax = plt.subplots(figsize=(6, 6))

            for idx in range(n_splits):
                interp_tpr = np.interp(mean_fpr, fprs[idx], tprs[idx])
                interp_tpr[0] = 0.0
                interp_tprs.append(interp_tpr)

            mean_tpr = np.mean(interp_tprs, axis=0)
            mean_tpr[-1] = 1.0
            mean_auc = auc(mean_fpr, mean_tpr)
            std_auc = np.std(roc_aucs)

            ax.plot(
                mean_fpr,
                mean_tpr,
                label=f"Mean ROC (AUC = {mean_auc:.3f} "+ r'$\pm$' + f" {std_auc:.3f})",
                lw=1,
                alpha=0.8
            )
            plt.grid(axis='both')
            std_tpr = np.std(interp_tprs, axis=0)
            tprs_upper = np.minimum(mean_tpr + std_tpr, 1)
            tprs_lower = np.maximum(mean_tpr - std_tpr, 0)
            ax.fill_between(
                mean_fpr,
                tprs_lower,
                tprs_upper,
                alpha=0.2,
                label=r"$\pm$ 1 std. dev.",
            )
            ax.set(
                xlabel="False Positive Rate",
                ylabel="True Positive Rate",
                title=f"{model_name} ROC - {t}, hrs{w}\nAUC = {mean_auc:.3f} "+ r'$\pm$' + f" {std_auc:.3f}\nAcc = {mean_acc:.3f} " + r'$\pm$' + f" {std_acc:.3f}",
            )

            plt.plot([0, 1], [0, 1], "k--", lw=1)
            ax.legend(loc="lower right")

            # save the figure
            parent_dir = f"results/figures/time_specific/{short}"
            os.makedirs(parent_dir, exist_ok=True)
            out_path = f"{parent_dir}/roc_{short}_{t}_hrs{w}.png"
            plt.savefig(out_path, dpi=150, bbox_inches="tight")
            plt.close()

            # SAVE THE BEST MODELS
            ## Find and train best (according to ROC AUC) classifier from cross-validation
            best_fold = np.argmax(roc_aucs)
            (train_idx, test_idx) = fold_index[best_fold]
            X_train = X.iloc[train_idx]
            y_train = y.iloc[train_idx]

            cv_best_classifier = clone(classifier)
            cv_best_classifier.fit(X_train, y_train)

            cv_dir = f"results/models/time_specific/cv"
            cv_path = f"{cv_dir}/{short}_{t}_{w}.pkl"
            os.makedirs(cv_dir, exist_ok=True)
            pickle.dump(cv_best_classifier, open(cv_path, "wb"))

            ## Train a separate model, on all of the data
            all_data_classifier = clone(classifier)
            all_data_classifier.fit(X, y)

            all_data_dir = f"results/models/time_specific/all_data"
            all_data_path = f"{all_data_dir}/{short}_{t}_{w}.pkl"
            os.makedirs(all_data_dir, exist_ok=True)
            pickle.dump(all_data_classifier, open(all_data_path, "wb"))

            # Record info for per-window plot
            for key in ['mean_fpr', 'mean_tpr', 'std_tpr', 'tprs_upper', 'tprs_lower', 'mean_auc', 'mean_acc', 'std_auc', 'std_acc']:
                exec(f"tissue_plotting_info['{t}']['{key}'] = {key}")

            print(f"Trained {model_name} for {t} at hrs{w}: ROC curve saved at {out_path}")
        
        plt.figure(figsize=(8, 8))
        # Generate per-window combined plot
        fig, ax = plt.subplots(figsize=(6, 6))
        ax.set(
        xlabel="False Positive Rate",
        ylabel="True Positive Rate",
        title=f"Mean {model_name} ROC curves, hrs{w}",
        )
        for t in tissues:
            # 
            ## AAaAAaaaaaAA dlaczego poza funkcją działało?!?!?
            ## decode the dictionary back lol
            # for key in ['mean_fpr', 'mean_tpr', 'std_tpr', 'tprs_upper', 'tprs_lower', 'mean_auc', 'mean_acc', 'std_auc', 'std_acc']:
            #     exec(f"{key} = tissue_plotting_info['{t}']['{key}']")
            ## z jakiegoś powodu nadpisywanie zmiennych nie działa w ten sposób gdy jestem wewnątrz funkcji
            #
            # ax.plot(    
            #     mean_fpr,
            #     mean_tpr,
            #     label=f"Mean ROC for {t} (AUC = {mean_auc:.3f} "+ r'$\pm$'+ f" {std_auc:.3f})",
            #     lw=1,
            #     alpha=0.8
            # )
            #
            # plt.grid(axis='both')
            #
            # ax.fill_between(
            #     mean_fpr,
            #     tprs_lower,
            #     tprs_upper,
            #     alpha=0.2,
            # )
            ax.plot(    
                tissue_plotting_info[t]['mean_fpr'],
                tissue_plotting_info[t]['mean_tpr'],
                label=f"Mean ROC for {t} (AUC = {tissue_plotting_info[t]['mean_auc']:.3f} "+ r'$\pm$'+ f" {tissue_plotting_info[t]['std_auc']:.3f})",
                lw=1,
                alpha=0.8
            )

            plt.grid(axis='both')
            
            ax.fill_between(
                tissue_plotting_info[t]['mean_fpr'],
                tissue_plotting_info[t]['tprs_lower'],
                tissue_plotting_info[t]['tprs_upper'],
                alpha=0.2,
            )


        plt.plot([0, 1], [0, 1], "k--", lw=1)
        ax.legend(loc="lower right")   

        # save the figure
        parent_dir = f"results/figures/time_specific/{short}"
        os.makedirs(parent_dir, exist_ok=True)
        out_path = f"{parent_dir}/roc_{short}_combined_hrs{w}.png"
        plt.savefig(out_path, dpi=150, bbox_inches="tight")
        plt.close()

        print(f"Saved {model_name} ROC plot for window {w}: {out_path}")

### Random Forest classifiers

In [3]:
classifier = RandomForestClassifier(
                n_estimators=500,
                random_state=0,
                n_jobs=-1
            )

time_specific_with_cv_and_roc_plots(classifier=classifier)

Trained Random Forest for Neuroblasts at hrs06-08: ROC curve saved at results/figures/time_specific/RF/roc_RF_Neuroblasts_hrs06-08.png
Trained Random Forest for Neurons at hrs06-08: ROC curve saved at results/figures/time_specific/RF/roc_RF_Neurons_hrs06-08.png
Trained Random Forest for Glia at hrs06-08: ROC curve saved at results/figures/time_specific/RF/roc_RF_Glia_hrs06-08.png
Saved Random Forest ROC plot for window 06-08: results/figures/time_specific/RF/roc_RF_combined_hrs06-08.png
Trained Random Forest for Neuroblasts at hrs10-12: ROC curve saved at results/figures/time_specific/RF/roc_RF_Neuroblasts_hrs10-12.png
Trained Random Forest for Neurons at hrs10-12: ROC curve saved at results/figures/time_specific/RF/roc_RF_Neurons_hrs10-12.png
Trained Random Forest for Glia at hrs10-12: ROC curve saved at results/figures/time_specific/RF/roc_RF_Glia_hrs10-12.png
Saved Random Forest ROC plot for window 10-12: results/figures/time_specific/RF/roc_RF_combined_hrs10-12.png
Trained Random F

<Figure size 800x800 with 0 Axes>

<Figure size 800x800 with 0 Axes>

<Figure size 800x800 with 0 Axes>

### Support Vector Machine

In [4]:
classifier = SVC(probability=True)

time_specific_with_cv_and_roc_plots(classifier=classifier)

Trained Support Vector Machine for Neuroblasts at hrs06-08: ROC curve saved at results/figures/time_specific/SVM/roc_SVM_Neuroblasts_hrs06-08.png
Trained Support Vector Machine for Neurons at hrs06-08: ROC curve saved at results/figures/time_specific/SVM/roc_SVM_Neurons_hrs06-08.png
Trained Support Vector Machine for Glia at hrs06-08: ROC curve saved at results/figures/time_specific/SVM/roc_SVM_Glia_hrs06-08.png
Saved Support Vector Machine ROC plot for window 06-08: results/figures/time_specific/SVM/roc_SVM_combined_hrs06-08.png
Trained Support Vector Machine for Neuroblasts at hrs10-12: ROC curve saved at results/figures/time_specific/SVM/roc_SVM_Neuroblasts_hrs10-12.png
Trained Support Vector Machine for Neurons at hrs10-12: ROC curve saved at results/figures/time_specific/SVM/roc_SVM_Neurons_hrs10-12.png
Trained Support Vector Machine for Glia at hrs10-12: ROC curve saved at results/figures/time_specific/SVM/roc_SVM_Glia_hrs10-12.png
Saved Support Vector Machine ROC plot for window

<Figure size 800x800 with 0 Axes>

<Figure size 800x800 with 0 Axes>

<Figure size 800x800 with 0 Axes>

### Logistic Regression (unregularized)

In [5]:
classifier = LogisticRegression()

time_specific_with_cv_and_roc_plots(classifier=classifier)

Trained Logistic Regression for Neuroblasts at hrs06-08: ROC curve saved at results/figures/time_specific/LR/roc_LR_Neuroblasts_hrs06-08.png
Trained Logistic Regression for Neurons at hrs06-08: ROC curve saved at results/figures/time_specific/LR/roc_LR_Neurons_hrs06-08.png
Trained Logistic Regression for Glia at hrs06-08: ROC curve saved at results/figures/time_specific/LR/roc_LR_Glia_hrs06-08.png
Saved Logistic Regression ROC plot for window 06-08: results/figures/time_specific/LR/roc_LR_combined_hrs06-08.png
Trained Logistic Regression for Neuroblasts at hrs10-12: ROC curve saved at results/figures/time_specific/LR/roc_LR_Neuroblasts_hrs10-12.png
Trained Logistic Regression for Neurons at hrs10-12: ROC curve saved at results/figures/time_specific/LR/roc_LR_Neurons_hrs10-12.png
Trained Logistic Regression for Glia at hrs10-12: ROC curve saved at results/figures/time_specific/LR/roc_LR_Glia_hrs10-12.png
Saved Logistic Regression ROC plot for window 10-12: results/figures/time_specific/

<Figure size 800x800 with 0 Axes>

<Figure size 800x800 with 0 Axes>

<Figure size 800x800 with 0 Axes>

### XGBoost

In [6]:
classifier = XGBClassifier(
    n_estimators = 500,
    n_jobs = -1
)

time_specific_with_cv_and_roc_plots(classifier=classifier)

Trained XGBoost for Neuroblasts at hrs06-08: ROC curve saved at results/figures/time_specific/XGB/roc_XGB_Neuroblasts_hrs06-08.png
Trained XGBoost for Neurons at hrs06-08: ROC curve saved at results/figures/time_specific/XGB/roc_XGB_Neurons_hrs06-08.png
Trained XGBoost for Glia at hrs06-08: ROC curve saved at results/figures/time_specific/XGB/roc_XGB_Glia_hrs06-08.png
Saved XGBoost ROC plot for window 06-08: results/figures/time_specific/XGB/roc_XGB_combined_hrs06-08.png
Trained XGBoost for Neuroblasts at hrs10-12: ROC curve saved at results/figures/time_specific/XGB/roc_XGB_Neuroblasts_hrs10-12.png
Trained XGBoost for Neurons at hrs10-12: ROC curve saved at results/figures/time_specific/XGB/roc_XGB_Neurons_hrs10-12.png
Trained XGBoost for Glia at hrs10-12: ROC curve saved at results/figures/time_specific/XGB/roc_XGB_Glia_hrs10-12.png
Saved XGBoost ROC plot for window 10-12: results/figures/time_specific/XGB/roc_XGB_combined_hrs10-12.png
Trained XGBoost for Neuroblasts at hrs14-16: ROC

<Figure size 800x800 with 0 Axes>

<Figure size 800x800 with 0 Axes>

<Figure size 800x800 with 0 Axes>

## Time-agnostic classifiers

### This section trains time-agnostic (3 tissues) classifiers for chromatin loop formation prediction

1. Trains cross-validated classifiers and plots their ROC curves (with mean and std)
2. Generates a combined plot with ROC curves of the best CV classifier for each tissue
3. For each tissue it saves two models:
    - best cross-validation classifier
    - classifier fitted to the entire data
4. For each tissue: compares the performance of different architectures

### Brudnopis
Funkcja(`klasyfikator`):
1. Dla tkanki: ładuje dataframe ze wszystkich trzech okien
2. łączenie i etykietowanie krotkami (okno, klasa)
3. Potem chyba wszystko tak samo, tylko że ze `StratifiedKFold`
 - czyli wykres z CV dla każdej tkanki (3 osobne)
 - jeden wspólny wykres dla tkanek (1 wykres, trzy krzywe) <- *słownik!!!*
 - mogę zrobić `json.dump` tego słownika dziwnego i wczytać go w drugiej analizie *WOW*
4. Będę nadal chciał mieć ten tkankowy słownik do plotowania, tylko teraz wykres combined będzie **dla architektury** a trzy krzywe będą pochodzić **od tkanek**

Potem, co nawet ważniejsze, porównujemy architektury wewnątrz tkanki. To będą 3 wykresy po 4 krzywe.
 - chcemy mieć mean i std z CV, więc nie możemy po prostu wczytywać modelu z .pkl
 - wczytać `json` jako słownik plotowania (kluczami są tkanki, głębszymi kluczami są poszczególne zmienne) --> wtedy mam jeden `json` dla jednej architektury

```{python}
for t in tissues:
    wczytuje słownik modelowy[t]
    robię wykres dla tkanki
```


dodać combined pomiędzy tkankami na końcu tej funkcji!

In [3]:
def time_agnostic_with_cv_and_roc_plots(classifier, windows=["06-08", "10-12", "14-16"], tissues=["Neuroblasts", "Neurons", "Glia"], n_splits=10):
    """
    Train tissue-specific, time-agnostic, cross-validated classifiers and save them. Generate ROC curve for each tissue with CV mean and std.
    args:
        classifier: one of a few predefined sklearn classifier objects
    returns:
        tissue_plotting_info: dictionary with information for downstream analyses
    """
    def make_names_dict():
        # FIXME: move this to some kind of config
        # Dictionary for f-string construction in model paths and in text on ROC plots
        model_names = ['RandomForestClassifier', 'SVC', 'LogisticRegression', 'XGBClassifier']
        model_names_dict = {name:{'full':'', 'short':''} for name in model_names}

        model_names_dict['RandomForestClassifier']['full'] = 'Random Forest'
        model_names_dict['RandomForestClassifier']['short'] = 'RF'

        model_names_dict['SVC']['full'] = 'Support Vector Machine'
        model_names_dict['SVC']['short'] = 'SVM'

        model_names_dict['LogisticRegression']['full'] = 'Logistic Regression'
        model_names_dict['LogisticRegression']['short'] = 'LR'

        model_names_dict['XGBClassifier']['full'] = 'XGBoost'
        model_names_dict['XGBClassifier']['short'] = 'XGB'

        return model_names_dict
    
    def compose_windows(tissue, windows=windows):
        """
        concatenate window-specific DataFrames and generate a `composite` vector for stratification
        """
        Xs, ys = [], []
        for idx, w in enumerate(windows):
            curr_X = pd.read_csv(f"data/training/hrs{w}/data_diff_hrs{w}.csv", index_col=0)
            curr_y = pd.read_csv(f"data/training/hrs{w}/y_{tissue}.csv", index_col=0).iloc[:, 0]
            curr_X['window'] = idx

            Xs.append(curr_X)
            ys.append(curr_y)

        y_new = pd.concat(ys, axis=0)
        X_new = pd.concat(Xs, axis=0)

        composite = pd.Categorical(list(zip(X_new['window'], y_new))).codes
        num_values = len(pd.Series(composite).unique())
        print(f"Created a composite vector with {num_values} distinct values")

        X_new.drop('window', axis=1, inplace=True) # we don't want to use 'window' for prediction

        return X_new, y_new, composite

    model_names_dict = make_names_dict()
    model_name = model_names_dict[type(classifier).__name__]['full']
    short = model_names_dict[type(classifier).__name__]['short']
    
    tissue_plotting_info = {t: {} for t in tissues}

    for t in tissues:
        # Load windows
        (X, y, composite) = compose_windows(t)

        # Align X and y just in case
        #X = X.loc[y.index]

        # Cross-validation
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=0)
        probs, trues, accs = [], [], []
        fold_index = {}

        for (i, (train_idx, test_idx)) in enumerate(skf.split(X, composite)):
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
            fold_index[i] = (train_idx, test_idx)

            classifier = clone(classifier) # this creates an UNFITTED model with the same parameters as before
            classifier.fit(X_train, y_train)

            p = classifier.predict_proba(X_test)[:, 1]
            probs.append(p)
            trues.append(y_test.values)
            accs.append(accuracy_score(y_test, (p > 0.5).astype(int)))

        # Compute ROC curve and other metrics
        mean_acc = np.mean(accs)
        std_acc = np.std(accs)

        fprs, tprs, roc_aucs = [], [], []
        for (true, prob) in zip(trues, probs):
            fpr, tpr, _ = roc_curve(true, prob)
            fprs.append(fpr)
            tprs.append(tpr)
            roc_aucs.append(auc(fpr, tpr))

        # Use linear interpolation to plot mean +/- 1 std. bounds
        mean_fpr = np.linspace(0, 1, 100)
        interp_tprs = []

        _, ax = plt.subplots(figsize=(6, 6))

        for idx in range(n_splits):
            interp_tpr = np.interp(mean_fpr, fprs[idx], tprs[idx])
            interp_tpr[0] = 0.0
            interp_tprs.append(interp_tpr)

        mean_tpr = np.mean(interp_tprs, axis=0)
        mean_tpr[-1] = 1.0
        mean_auc = auc(mean_fpr, mean_tpr)
        std_auc = np.std(roc_aucs)

        ax.plot(
            mean_fpr,
            mean_tpr,
            label=f"Mean ROC (AUC = {mean_auc:.3f} "+ r'$\pm$' + f" {std_auc:.3f})",
            lw=1,
            alpha=0.8
        )
        plt.grid(axis='both')
        std_tpr = np.std(interp_tprs, axis=0)
        tprs_upper = np.minimum(mean_tpr + std_tpr, 1)
        tprs_lower = np.maximum(mean_tpr - std_tpr, 0)
        ax.fill_between(
            mean_fpr,
            tprs_lower,
            tprs_upper,
            alpha=0.2,
            label=r"$\pm$ 1 std. dev.",
        )
        ax.set(
            xlabel="False Positive Rate",
            ylabel="True Positive Rate",
            title=f"Time-agnostic {model_name} ROC ({t})\nAUC = {mean_auc:.3f} "+ r'$\pm$' + f" {std_auc:.3f}\nAcc = {mean_acc:.3f} " + r'$\pm$' + f" {std_acc:.3f}",
        )

        plt.plot([0, 1], [0, 1], "k--", lw=1)
        ax.legend(loc="lower right")

        # save the figure
        parent_dir = f"results/figures/time_agnostic/{short}"
        os.makedirs(parent_dir, exist_ok=True)
        out_path = f"{parent_dir}/roc_{short}_{t}.png"
        plt.savefig(out_path, dpi=150, bbox_inches="tight")
        plt.close()

        # SAVE THE BEST MODELS
        ## Find and train best (according to ROC AUC) classifier from cross-validation
        best_fold = np.argmax(roc_aucs)
        (train_idx, test_idx) = fold_index[best_fold]
        X_train = X.iloc[train_idx]
        y_train = y.iloc[train_idx]

        cv_best_classifier = clone(classifier)
        cv_best_classifier.fit(X_train, y_train)

        cv_dir = f"results/models/time_agnostic/cv"
        cv_path = f"{cv_dir}/{short}_{t}.pkl"
        os.makedirs(cv_dir, exist_ok=True)
        pickle.dump(cv_best_classifier, open(cv_path, "wb"))

        ## Train a separate model, on all of the data
        all_data_classifier = clone(classifier)
        all_data_classifier.fit(X, y)

        all_data_dir = f"results/models/time_agnostic/all_data"
        all_data_path = f"{all_data_dir}/{short}_{t}.pkl"
        os.makedirs(all_data_dir, exist_ok=True)
        pickle.dump(all_data_classifier, open(all_data_path, "wb"))

        # IMPORTANT!
        # Record info for per-window plot
        for key in ['mean_fpr', 'mean_tpr', 'tprs_upper', 'tprs_lower', 'mean_auc', 'mean_acc', 'std_auc', 'std_acc']:
            exec(f"tissue_plotting_info['{t}']['{key}'] = {key}")

        print(f"Trained {model_name} for {t}: ROC curve saved at {out_path}")

    # FIXME: share this info in a more sensible way (like json dump)
    return tissue_plotting_info
"""
# PER WINDOW: -----------------------------------------------------------------
        plt.figure(figsize=(8, 8))
        # Generate per-window combined plot
        fig, ax = plt.subplots(figsize=(6, 6))
        ax.set(
        xlabel="False Positive Rate",
        ylabel="True Positive Rate",
        title=f"Mean {model_name} ROC curves, hrs{w}",
        )
        for t in tissues:
            # 
            ## AAaAAaaaaaAA dlaczego poza funkcją działało?!?!?
            ## decode the dictionary back lol
            # for key in ['mean_fpr', 'mean_tpr', 'std_tpr', 'tprs_upper', 'tprs_lower', 'mean_auc', 'mean_acc', 'std_auc', 'std_acc']:
            #     exec(f"{key} = tissue_plotting_info['{t}']['{key}']")
            ## z jakiegoś powodu nadpisywanie zmiennych nie działa w ten sposób gdy jestem wewnątrz funkcji
            #
            # ax.plot(    
            #     mean_fpr,
            #     mean_tpr,
            #     label=f"Mean ROC for {t} (AUC = {mean_auc:.3f} "+ r'$\pm$'+ f" {std_auc:.3f})",
            #     lw=1,
            #     alpha=0.8
            # )
            #
            # plt.grid(axis='both')
            #
            # ax.fill_between(
            #     mean_fpr,
            #     tprs_lower,
            #     tprs_upper,
            #     alpha=0.2,
            # )
            ax.plot(    
                tissue_plotting_info[t]['mean_fpr'],
                tissue_plotting_info[t]['mean_tpr'],
                label=f"Mean ROC for {t} (AUC = {tissue_plotting_info[t]['mean_auc']:.3f} "+ r'$\pm$'+ f" {tissue_plotting_info[t]['std_auc']:.3f})",
                lw=1,
                alpha=0.8
            )

            plt.grid(axis='both')
            
            ax.fill_between(
                tissue_plotting_info[t]['mean_fpr'],
                tissue_plotting_info[t]['tprs_lower'],
                tissue_plotting_info[t]['tprs_upper'],
                alpha=0.2,
            )


        plt.plot([0, 1], [0, 1], "k--", lw=1)
        ax.legend(loc="lower right")   

        # save the figure
        parent_dir = f"results/figures/time_specific/{short}"
        os.makedirs(parent_dir, exist_ok=True)
        out_path = f"{parent_dir}/roc_{short}_combined_hrs{w}.png"
        plt.savefig(out_path, dpi=150, bbox_inches="tight")
        plt.close()

        print(f"Saved {model_name} ROC plot for window {w}: {out_path}")
"""

<>:199: SyntaxWarning: invalid escape sequence '\p'
<>:199: SyntaxWarning: invalid escape sequence '\p'
/tmp/ipykernel_9297/2791450959.py:199: SyntaxWarning: invalid escape sequence '\p'
  #     label=f"Mean ROC for {t} (AUC = {mean_auc:.3f} "+ r'$\pm$'+ f" {std_auc:.3f})",


'\n# PER WINDOW: -----------------------------------------------------------------\n        plt.figure(figsize=(8, 8))\n        # Generate per-window combined plot\n        fig, ax = plt.subplots(figsize=(6, 6))\n        ax.set(\n        xlabel="False Positive Rate",\n        ylabel="True Positive Rate",\n        title=f"Mean {model_name} ROC curves, hrs{w}",\n        )\n        for t in tissues:\n            # \n            ## AAaAAaaaaaAA dlaczego poza funkcją działało?!?!?\n            ## decode the dictionary back lol\n            # for key in [\'mean_fpr\', \'mean_tpr\', \'std_tpr\', \'tprs_upper\', \'tprs_lower\', \'mean_auc\', \'mean_acc\', \'std_auc\', \'std_acc\']:\n            #     exec(f"{key} = tissue_plotting_info[\'{t}\'][\'{key}\']")\n            ## z jakiegoś powodu nadpisywanie zmiennych nie działa w ten sposób gdy jestem wewnątrz funkcji\n            #\n            # ax.plot(    \n            #     mean_fpr,\n            #     mean_tpr,\n            #     label=f"Mea

### Time-agnostic **Random Forest**

In [4]:
classifier = RandomForestClassifier(
                n_estimators=500,
                random_state=0,
                n_jobs=-1
            )

rf_dict = time_agnostic_with_cv_and_roc_plots(classifier=classifier)

Created a composite vector with 6 distinct values
Trained Random Forest for Neuroblasts: ROC curve saved at results/figures/time_agnostic/RF/roc_RF_Neuroblasts.png
Created a composite vector with 6 distinct values
Trained Random Forest for Neurons: ROC curve saved at results/figures/time_agnostic/RF/roc_RF_Neurons.png
Created a composite vector with 6 distinct values
Trained Random Forest for Glia: ROC curve saved at results/figures/time_agnostic/RF/roc_RF_Glia.png


### Time-agnostic **Support Vector Machine**

In [5]:
classifier = SVC(probability=True)

rf_dict = time_agnostic_with_cv_and_roc_plots(classifier=classifier)

Created a composite vector with 6 distinct values
Trained Support Vector Machine for Neuroblasts: ROC curve saved at results/figures/time_agnostic/SVM/roc_SVM_Neuroblasts.png
Created a composite vector with 6 distinct values
Trained Support Vector Machine for Neurons: ROC curve saved at results/figures/time_agnostic/SVM/roc_SVM_Neurons.png
Created a composite vector with 6 distinct values
Trained Support Vector Machine for Glia: ROC curve saved at results/figures/time_agnostic/SVM/roc_SVM_Glia.png


### Time-agnostic **Logistic Regression** (unregularized)

In [6]:
classifier = LogisticRegression()

lr_dict = time_agnostic_with_cv_and_roc_plots(classifier=classifier)

Created a composite vector with 6 distinct values
Trained Logistic Regression for Neuroblasts: ROC curve saved at results/figures/time_agnostic/LR/roc_LR_Neuroblasts.png
Created a composite vector with 6 distinct values
Trained Logistic Regression for Neurons: ROC curve saved at results/figures/time_agnostic/LR/roc_LR_Neurons.png
Created a composite vector with 6 distinct values
Trained Logistic Regression for Glia: ROC curve saved at results/figures/time_agnostic/LR/roc_LR_Glia.png


### Time-agnostic **XGBoost**

In [7]:
classifier = XGBClassifier(
    n_estimators = 500,
    n_jobs = -1
)

xgb_dict = time_agnostic_with_cv_and_roc_plots(classifier=classifier)

Created a composite vector with 6 distinct values
Trained XGBoost for Neuroblasts: ROC curve saved at results/figures/time_agnostic/XGB/roc_XGB_Neuroblasts.png
Created a composite vector with 6 distinct values
Trained XGBoost for Neurons: ROC curve saved at results/figures/time_agnostic/XGB/roc_XGB_Neurons.png
Created a composite vector with 6 distinct values
Trained XGBoost for Glia: ROC curve saved at results/figures/time_agnostic/XGB/roc_XGB_Glia.png


In [ ]:
xgb_dict

## Grouped barplot z wąsami robimy

In [ ]:
# sprawdzić klucze w dictach (czy są wszystkie tkanki)

In [ ]:
def make_barplot_from_dicts():
    pass

# def make_barplot_from_jsons():
#     pass